# TRANSLATION TEST

Try translating different types of texts (simple sentences, multiple sentences with bullet points, etc.) from various languages ("eu", "en") using the `Elia.eus` translator.

In [87]:
import requests

def correct_response_text(translated_sentences):
    """
    Function that processes a list of translated sentences, correcting formatting issues by merging sentence fragments.
    
    Parameters:
        translated_sentences (list): A list of strings representing translated sentences.
    
    Returns:
        str: A formatted string where certain sentences are merged for better readability.
    """
    corrected_sentences = []
    ctrl = False

    # Iterate over the translated sentences
    for i in range(len(translated_sentences)):
        # Check if the sentence starts with a number (e.g., '1.')
        if translated_sentences[i].strip().endswith('.'):
            # If the next sentence exists, join the current sentence with the next one
            if i + 1 < len(translated_sentences):
                corrected_sentences.append(translated_sentences[i].strip() + ' ' + translated_sentences[i + 1].strip())
                ctrl = True # mark to ignore next sentence as its is joined 
            else:
                corrected_sentences.append(translated_sentences[i].strip())
        elif ctrl:
             ctrl = False # reset control
             continue;
        else:
            # If not starting with a number, just add the sentence as it is
            corrected_sentences.append(translated_sentences[i].strip())
    
    # Join the sentences with a newline character
    result_string = '\n'.join(corrected_sentences)
    return result_string


def translate_elia_session(text, src_lang, dst_lang, verbose = False):
    """
    Function that translates a given text from a source language to a target language using the Elia translation service.

    Parameters:
        text (str): The text to be translated.
        src_lang (str): The source language code (e.g., 'eu' for Basque).
        dst_lang (str): The target language code (e.g., 'en' for English).
        verbose (bool, optional): If True, prints the response JSON for debugging. Default is False.

    Returns:
        str: The translated text after processing.
    """
    # URL of the main translation page (GET request to retrieve CSRF token and cookies)
    url = "https://elia.eus/traductor"  
    
    # URL for making the POST request to get the translated text
    post_url = "https://elia.eus/ajax/translate_string"  

    # Create a session to maintain cookies
    session = requests.Session()

    # Perform the initial GET request to get cookies and the CSRF token
    response = session.get(url)
    
    # Check if the request was successful
    if response.status_code != 200:
        return f"Error fetching the page: {response.status_code}"

    # Extract the CSRF token from the cookies (this can vary depending on the HTML structure)
    csrf_token = None
    for cookie in session.cookies:
        if cookie.name == "csrftoken":  # Look for the csrf token in the cookies
            csrf_token = cookie.value
            break

    # If no CSRF token is found, return an error
    if not csrf_token:
        return "CSRF token not found."

    # Prepare the headers for the POST request, specifying content type, and referring origin
    headers = {
        "Accept": "application/json, text/javascript, */*; q=0.01",  # Accepting JSON response
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",  # Content type for form submission
        "X-Requested-With": "XMLHttpRequest",  # Indicate the request is an AJAX request
        "Origin": "https://elia.eus",  # Origin header (same as the website)
        "Referer": url,  # Referer header to indicate the source of the request
    }

    # Prepare the payload (data) for the POST request
    data = {
        'csrfmiddlewaretoken': csrf_token,  # CSRF token to prevent cross-site request forgery
        'source_language': src_lang,  # Source language code (e.g., 'eu' for Basque)
        'input_text': text,  # The text to be translated
        'translation_engine': '1',  # The translation engine (usually a default value)
        'target_language': dst_lang,  # Target language code (e.g., 'es' for Spanish)
        'translation_model': 'general',  # The translation model (e.g., 'general' translation)
        'target_voice': 'F',  # Voice for the target language, 'F' for female and 'M' for male
    }

    # Perform the POST request to get the translated text
    response = session.post(post_url, data=data, headers=headers)
    
    # Check if the POST request was successful
    if response.status_code == 200:
        result = response.json()  # Parse the response as json     
        if verbose:
            print(result)
        return correct_response_text(result['translated_sentences']) #get translated sentences and compound string
    else:
        return f"Error translating: {response.status_code}"  # Return an error if translation fails

    
# Translation examples:
input_text_eu = "azaldu adimen artifiziala"
translated_text_en =  translate_elia_session(input_text_eu, "eu", "en")
print(f"Text in Basque: {input_text_eu}")
print(f"Translated text in English: {translated_text_en}")

# Translation examples:
input_text_en = """Here are some short and delicious light dinner ideas:

1. Grilled chicken or fish with roasted vegetables
2. Salad with protein of your choice (e.g., grilled chicken, salmon, tofu)
3. Lentil soup with whole grain bread
4. Stuffed bell peppers with quinoa and veggies
5. Omelette with veggies and cheese
6. Fresh fruit salad with yogurt or granola
7. Mini paninis or wraps with lean meats and veggies

Which one sounds appealing to you?"""
translated_text_eu =  translate_elia_session(input_text_en, "en", "eu")
print(f"----\nText in English: {input_text_en}")
print(f"Translated text in Basque: {translated_text_eu}")

Text in Basque: azaldu adimen artifiziala
Translated text in English: Explain artificial intelligence
----
Text in English: Here are some short and delicious light dinner ideas:

1. Grilled chicken or fish with roasted vegetables
2. Salad with protein of your choice (e.g., grilled chicken, salmon, tofu)
3. Lentil soup with whole grain bread
4. Stuffed bell peppers with quinoa and veggies
5. Omelette with veggies and cheese
6. Fresh fruit salad with yogurt or granola
7. Mini paninis or wraps with lean meats and veggies

Which one sounds appealing to you?
Translated text in Basque: Hona hemen afari arina egiteko ideia labur eta goxo batzuk:
1. Oilaskoa edo arraina plantxan barazki erreekin
2. Entsalada nahi duzun proteinarekin (adibidez, oilaskoa plantxan, izokina, tofua)
3. Dilista-zopa osoko ogiarekin
4. Kanpai-piper beteak, kinoa eta barazkiekin
5. Tortilla begiekin eta gaztarekin
6. Fruta freskoen entsalada jogurtarekin edo granolarekin
7. Panina txikiak edo haragi gihartsu eta beget

# MODEL PULLING

Pull or download the `llama3.2:1b` model from the Ollama library.

In [2]:
# download model to fine-tune
import ollama

ollama.pull("llama3.2:1b")

ProgressResponse(status='success', completed=None, total=None, digest=None)

# MODEL ITERATION EXAMPLE

Ask a question to the LLM model in Basque, translate it to English with the `Elia.eus` translator, and display the result again in Basque.

In [88]:
# Test if model is working correctly with translation
import requests
from langchain.llms import Ollama

def truncate_text_to_100_words(text):
    """
    Function that truncates a given text to a maximum of 100 words, ensuring it ends at the last complete sentence if possible.
    
    Parameters:
        text (str): The input text to be truncated.
    
    Returns:
        str: The truncated text with a maximum of 100 words, ending at the last detected period if available.
    """
    
    # Split the text into a list of words
    words = text.split()

    # If the text has more than 100 words, truncate it
    if len(words) > 100:
        # Get the first 100 words
        truncated_text = " ".join(words[:100])

        # Now find the position of the last period (".") within the first 100 words
        last_period = truncated_text.rfind(".")

        # If a period is found, truncate at that position to end the sentence
        if last_period != -1:
            truncated_text = truncated_text[:last_period + 1]  # Include the period
        else:
            # If no period is found, just return the first 100 words
            truncated_text = " ".join(words[:100])
        
        return truncated_text
    else:
        # If the text has 100 words or fewer, return it as is
        return text
        
# prompt text
input_text = "Adimen artifiziala zer da? erantzun labur"

# Translate user text into english
translated_input_text = translate_elia_session(input_text, "eu", "en")

# Load Ollama model
llm = Ollama(model="llama3.2:1b")

# Ask to the model with tranlated text
response = llm.invoke(translated_input_text)
#print(response)

# truncate response to 100 words max
truncated_response = truncate_text_to_100_words(response)
#print(f"Truncated Response: {truncated_response}")

# check translations
print("<USER>:", input_text)
print("<MODEL>:", translate_elia_session(truncated_response, "en", "eu"))

<USER>: Adimen artifiziala zer da? erantzun labur
<MODEL>: Adimen artifizialak (IA) giza adimena behar ohi duten zereginak egin ditzaketen sistema informatikoen garapenari egiten dio erreferentzia, hala nola ikaskuntza, arazoen ebazpena, erabakiak hartzea eta pertzepzioa.


# CREATE WELCOME PROMPT TEMPLATE

Set the LLM model behavior in a simple way, translate it to English with the `Elia.eus` translator, and display the result if understood correctly. Answer again in Basque.

In [89]:
from langchain.prompts import PromptTemplate
from langchain.llms import Ollama

# Create template with instructions of behaviour as welcome message
prompt = PromptTemplate(
     template="""
        Sukaldean espezializatutako laguntzaile birtuala zara. Labur erantzuten duena (gehienez 100 hitz). Erabiltzailaren gustu zein azalpenen arabera aukera desberdinak proposatuko dituzu era garbi, motz eta zehatzean. Kaixo!
        """
)

# load ollama model
llm = Ollama(model="llama3.2:1b")

# process prompt template
formatted_prompt = prompt.format()

# Translate welcome message into english
translated_input_text = translate_elia_session(formatted_prompt, "eu", "en")
#print(translated_input_text)

# invoque welcome message passing message in english
response = llm.invoke(translated_input_text)

# truncate response to 100 words max
truncated_response = truncate_text_to_100_words(response)

# Get response in basque
print(f"<MODEL>: {translate_elia_session(truncated_response, 'en', 'eu')}")

<MODEL>: Zure sukalde-inguruarekin laguntzera nator. Zer dago menuan?
Sukaldaritza edo errezeta jakinen bat duzu buruan?
Edo agian zerbait orokorragoa?
Aukera sinple baina goxo batzuk eskain ditzaket, gustu eta behar dietetiko sorta bati erantzuten diotenak. Abisatu, eta neurrira egindako proposamen bat emango dut!


# CREATE QUESTION - ANSWERS CHAIN

Simulate questions and answers chain. Starting from a welcome message and continuing with question answer form.

In [98]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Create template for the welcome message
welcome_prompt = PromptTemplate(
   template="""
        Sukaldean espezializatutako laguntzaile birtuala zara. Labur erantzuten duena (gehienez 100 hitz). Erabiltzailaren gustu zein azalpenen arabera aukera desberdinak proposatuko dituzu era garbi, motz eta zehatzean. Kaixo!
        """
)

# Create template for the question prompt
translated_template = translate_elia_session("Mesedez, erantzun galdera honi argi eta labur : {question}", "eu", "en")

# Create the template for the question prompt using the translated string
question_prompt = PromptTemplate(
    input_variables=["question"],
    template=translated_template  # Use the translated template string here
)

# Load Ollama model
llm = Ollama(model="llama3.2:1b")

# Start conversation with the welcome message
welcome_message = welcome_prompt.format()
response = llm.invoke(translate_elia_session(welcome_message, "eu", "en"))

# Print the welcome message (in Basque)
print("<MODEL>:", translate_elia_session(response, "en", "eu"))

# Proceed with the first question after the welcome message
chain1 = LLMChain(llm=llm, prompt=question_prompt)

# Define the first question
question1 = "Kaixo, proposatu afari plater arin bat."

# Ask the first question (in Basque, translated to English for the model)
response1 = chain1.run(translate_elia_session(question1, "eu", "en"))

# Print the response to the first question in Basque
print("<USER>: ", question1)
print("<MODEL>:", translate_elia_session(response1, "en", "eu"))

<MODEL>: Zure sukaldeko galderekin laguntzera nator. Zertan lagun zaitzaket gaur?
Errezeten iradokizunik, sukalderako aholkurik edo bestelakorik behar duzu?
Esadazu, eta ahal dudan guztia egingo dut 100 hitz edo gutxiagoko epean erantzun labur eta lagungarria emateko.
<USER>:  Kaixo, proposatu afari plater arin bat.
<MODEL>: Afaltzeko plater arin bat egiteko aukera sinple baina dotorea izan liteke:
* Lurrunetan egositako barazkien errazio txiki bat (adibidez, brokolia, azenarioa), oliba olio zorrotada batekin
* Pasta egosiko entsalada zati txiki bat (adibidez, pennea edo kaikua)
* Txingarretan erretako oilasko edo izokin xerra txiki bat, kinoa edo arroz integrala alde batekin
* Hummus edo guakamolezko trikuharri txiki bat, murgiltze gisa
Aukera horiek erraz prestatzen dira, gutxieneko osagaiak behar dituzte, eta hainbat gustu eta lehentasun dietetikora egokitzeko pertsonaliza daitezke.


# ADD MEMORY TO THE ITERATION (CONVERSATIONAL CHAIN)

Simulate a conversational chain with memory, starting with a welcome message and continuing in a question-and-answer format while referring to previous answers.

In [107]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.memory import ConversationBufferMemory
from langchain.llms import Ollama

# Create memory for the conversational chain
memory = ConversationBufferMemory(input_key="question", memory_key="chat_history")

# Create template for the welcome message
welcome_prompt = PromptTemplate(
    template="""
        Sukaldean espezializatutako laguntzaile birtuala zara. Labur erantzuten duena (gehienez 100 hitz). 
        Erabiltzailaren gustu zein azalpenen arabera aukera desberdinak proposatuko dituzu era garbi, motz eta zehatzean.
        Kaixo! Zer laguntza behar duzu gaur?
        """
)

# Create template for History Memory Questions
translated_template = translate_elia_session("""
        Aurreko elkarrizketa: {chat_history}
        Erabiltzaileak galdetu du: {question}
        Mesedez, erantzun argi eta labur.
        """, "eu", "en")
question_prompt = PromptTemplate(
    input_variables=["chat_history", "question"],
    template=translated_template
)

# Load Ollama model
llm = Ollama(model="llama3.2:1b")

# Start conversation with the welcome message
welcome_message = welcome_prompt.format()
response = llm.invoke(translate_elia_session(welcome_message, "eu", "en"))

print("<MODEL>:", translate_elia_session(response, "en", "eu"))

# Create the chain with memory
conversation_chain = LLMChain(llm=llm, prompt=question_prompt, memory=memory)

# First question
question1 = "Kaixo, proposatu afarirako plater arin bat. Bakarra mesedez."
response1 = conversation_chain.run(question=translate_elia_session(question1, "eu", "en"))

print("<USER>:", question1)
print("<MODEL>:", translate_elia_session(response1, "en", "eu"))

# Second question related to the first
question2 = "Eta, plater horretarako, zein saltsa gomendatzen duzu?"
response2 = conversation_chain.run(question=translate_elia_session(question2, "eu", "en"))

print("<USER>:", question2)
print("<MODEL>:", translate_elia_session(response2, "en", "eu"))


<MODEL>: Zure sukalde-beharrei laguntzera nator. Zertan lagun zaitzaket gaur?
Errezeten iradokizunik, sukaldaritzaren gidaritzarik edo, agian, osagaien ideiaren bat behar duzu?
Esadazu, eta proposamen labur bat emango dizut, zure gustuetara egokitua!
<USER>: Kaixo, proposatu afarirako plater arin bat. Bakarra mesedez.
<MODEL>: Zer moduz izokina plantxan limoi eta belarrekin?
Plater sinplea eta zaporetsua da, 30 minutu baino gutxiagoan prest dagoena eta barazki erreen edo kinoaren alde batekin zerbitzatu daitekeena.
<USER>: Eta, plater horretarako, zein saltsa gomendatzen duzu?
<MODEL>: Izokina limoi eta belarrekin plantxan egiteko, limoi-gurinezko saltsa sinple bat gomendatzen dizuet, gurin bigunduko 2 koilarakada limoi-zuku egin berriko koilarakada batekin eta baratxuri xehatuko iltze batekin konbinatuz egina. Horrek zapore goxoa eta ukitzailea erantsiko dio platerari, gehiegi indartu gabe.
Horrek zapore goxoa eta ukitzailea erantsiko dio platerari, gehiegi indartu gabe.
